
# Final High-Score Traffic Prediction Notebook

This notebook includes:

- Strong geo-temporal feature engineering
- OOF target encoding
- Rolling time-series folds
- Multi-fold CatBoost ensemble
- Pseudo labeling framework
- Competition-grade validation setup

Optimized for pushing beyond the 0.882 plateau.


In [20]:

# =========================
# INSTALLS
# =========================

# Uncomment if needed

# !pip install catboost pygeohash


In [21]:

# =========================
# IMPORTS
# =========================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor

import pygeohash as pgh


In [22]:

# =========================
# LOAD DATA
# =========================

train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')
sample = pd.read_csv('dataset/sample_submission.csv')

print(train.shape)
print(test.shape)


(77299, 11)
(41778, 10)


In [23]:

# =========================
# COMBINE DATA
# =========================

combined = pd.concat([
    train.drop(columns=['demand']),
    test
], axis=0).reset_index(drop=True)

print(combined.shape)


(119077, 10)


In [24]:

# =========================
# BASIC TYPE FIXES
# =========================

numeric_cols = [
    'day',
    'NumberofLanes',
    'Temperature'
]

for col in numeric_cols:

    combined[col] = pd.to_numeric(
        combined[col],
        errors='coerce'
    )

binary_cols = [
    'LargeVehicles',
    'Landmarks'
]

for col in binary_cols:

    combined[col] = combined[col].astype(str).str.lower()

    combined[col] = combined[col].map({
        'yes': 1,
        'no': 0,
        'true': 1,
        'false': 0,
        '1': 1,
        '0': 0
    })

    combined[col] = combined[col].fillna(0).astype(int)

print(combined.dtypes)


Index              int64
geohash              str
day                int64
timestamp            str
RoadType             str
NumberofLanes      int64
LargeVehicles      int64
Landmarks          int64
Temperature      float64
Weather              str
dtype: object


In [25]:

# =========================
# GEOHASH DECODING
# =========================

def decode_geohash(gh):

    try:

        decoded = pgh.decode_exactly(gh)

        return decoded[0], decoded[1]

    except:

        return np.nan, np.nan

combined[['latitude', 'longitude']] = combined['geohash'].apply(
    lambda x: pd.Series(decode_geohash(x))
)

combined.head()


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,latitude,longitude
0,0,qp02z1,48,0:0,NaN,1,0,0,NaN,NaN,-5.484924,90.664673
1,1,qp02zt,48,0:0,Residential,3,0,1,31.104565,Sunny,-5.462952,90.686646
2,2,qp08bj,48,0:0,Residential,1,0,0,25.919267,Sunny,-5.462952,90.708618
3,3,qp08gt,48,0:0,Residential,1,0,0,NaN,Rainy,-5.462952,90.862427
4,4,qp02zq,48,0:0,Residential,1,0,0,10.803667,Rainy,-5.457458,90.675659


In [26]:

# =========================
# GEOHASH HIERARCHY
# =========================

combined['geo_4'] = combined['geohash'].astype(str).str[:4]
combined['geo_5'] = combined['geohash'].astype(str).str[:5]
combined['geo_6'] = combined['geohash'].astype(str).str[:6]


In [27]:

# =========================
# TIME FEATURES
# =========================

combined['timestamp'] = pd.to_datetime(combined['timestamp'])

combined['hour'] = combined['timestamp'].dt.hour
combined['minute'] = combined['timestamp'].dt.minute
combined['dayofweek'] = combined['timestamp'].dt.dayofweek
combined['weekofyear'] = combined['timestamp'].dt.isocalendar().week.astype(int)

combined['minutes'] = (
    combined['hour'] * 60 +
    combined['minute']
)

combined['hour_sin'] = np.sin(
    2*np.pi*combined['hour']/24
)

combined['hour_cos'] = np.cos(
    2*np.pi*combined['hour']/24
)

combined['minute_sin'] = np.sin(
    2*np.pi*combined['minutes']/1440
)

combined['minute_cos'] = np.cos(
    2*np.pi*combined['minutes']/1440
)

combined['is_peak_morning'] = (
    combined['hour'].between(7,10)
).astype(int)

combined['is_peak_evening'] = (
    combined['hour'].between(16,20)
).astype(int)

combined['is_night'] = (
    combined['hour'].between(0,5)
).astype(int)

combined['is_weekend'] = (
    combined['dayofweek'].isin([5,6])
).astype(int)

combined.head()


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,weekofyear,minutes,hour_sin,hour_cos,minute_sin,minute_cos,is_peak_morning,is_peak_evening,is_night,is_weekend
0,0,qp02z1,48,1-01-01,NaN,1,0,0,NaN,NaN,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0
1,1,qp02zt,48,1-01-01,Residential,3,0,1,31.104565,Sunny,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0
2,2,qp08bj,48,1-01-01,Residential,1,0,0,25.919267,Sunny,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0
3,3,qp08gt,48,1-01-01,Residential,1,0,0,NaN,Rainy,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0
4,4,qp02zq,48,1-01-01,Residential,1,0,0,10.803667,Rainy,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0


In [28]:

# =========================
# INTERACTION FEATURES
# =========================

combined['temp_hour'] = (
    combined['Temperature'] *
    combined['hour']
)

combined['lanes_temp'] = (
    combined['NumberofLanes'] *
    combined['Temperature']
)

combined['vehicle_lane'] = (
    combined['LargeVehicles'] *
    combined['NumberofLanes']
)

combined['road_capacity'] = (
    combined['NumberofLanes'] *
    (combined['LargeVehicles'] + 1)
)

combined['road_weather'] = (
    combined['RoadType'].astype(str)
    + '_'
    + combined['Weather'].astype(str)
)


In [29]:

# =========================
# GEO TEMPORAL AGGREGATES
# =========================

train_len = len(train)

temp_train = combined.iloc[:train_len].copy()

temp_train['demand'] = train['demand'].values

geo_hour_mean = temp_train.groupby(
    ['geohash', 'hour']
)['demand'].mean()

geo_hour_std = temp_train.groupby(
    ['geohash', 'hour']
)['demand'].std()

geo_weather_mean = temp_train.groupby(
    ['geohash', 'Weather']
)['demand'].mean()

geo_day_mean = temp_train.groupby(
    ['geohash', 'day']
)['demand'].mean()

combined['geo_hour_mean'] = combined.set_index(
    ['geohash', 'hour']
).index.map(geo_hour_mean)

combined['geo_hour_std'] = combined.set_index(
    ['geohash', 'hour']
).index.map(geo_hour_std)

combined['geo_weather_mean'] = combined.set_index(
    ['geohash', 'Weather']
).index.map(geo_weather_mean)

combined['geo_day_mean'] = combined.set_index(
    ['geohash', 'day']
).index.map(geo_day_mean)


In [30]:

# =========================
# MISSING VALUE HANDLING
# =========================

numeric_columns = [
    col for col in combined.columns
    if pd.api.types.is_numeric_dtype(combined[col])
]

for col in numeric_columns:

    combined[col] = combined[col].fillna(
        combined[col].median()
    )

categorical_columns = [
    col for col in combined.columns
    if col not in numeric_columns
]

for col in categorical_columns:

    combined[col] = combined[col].astype(str)

    combined[col] = combined[col].fillna('Unknown')

print(combined.isnull().sum().sum())


0


In [31]:

# =========================
# SPLIT BACK
# =========================

train_df = combined.iloc[:len(train)].copy()
test_df = combined.iloc[len(train):].copy()

train_df['demand'] = train['demand'].values


In [32]:

# =========================
# OOF TARGET ENCODING
# =========================

def oof_target_encoding(
    train_df,
    test_df,
    column,
    target,
    n_splits=5
):

    oof = np.zeros(len(train_df))
    test_preds = np.zeros(len(test_df))

    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    for train_idx, val_idx in kf.split(train_df):

        X_tr = train_df.iloc[train_idx]
        X_val = train_df.iloc[val_idx]

        means = X_tr.groupby(column)[target].mean()

        oof[val_idx] = X_val[column].map(means)

        test_preds += test_df[column].map(means).fillna(
            X_tr[target].mean()
        ) / n_splits

    oof = pd.Series(oof).fillna(
        train_df[target].mean()
    )

    return oof, test_preds


In [33]:

# =========================
# APPLY TARGET ENCODING
# =========================

target_encode_cols = [
    'geohash',
    'geo_4',
    'geo_5',
    'RoadType',
    'Weather'
]

for col in target_encode_cols:

    train_df[f'{col}_te'], test_df[f'{col}_te'] = (
        oof_target_encoding(
            train_df,
            test_df,
            col,
            'demand'
        )
    )

print("OOF target encoding complete.")


OOF target encoding complete.


In [34]:

# =========================
# FEATURE LIST
# =========================

drop_cols = [
    'Index',
    'timestamp',
    'demand'
]

features = [
    col for col in train_df.columns
    if col not in drop_cols
]

categorical_features = [
    'geohash',
    'geo_4',
    'geo_5',
    'geo_6',
    'RoadType',
    'Weather',
    'road_weather'
]

categorical_features = [
    col for col in categorical_features
    if col in features
]

print(len(features))


40


In [35]:
# =========================
# SAFE TIME SERIES FOLDS
# =========================

unique_days = sorted(
    train_df['day'].dropna().unique()
)

print("Unique Days:", unique_days)
print("Total Unique Days:", len(unique_days))

n_days = len(unique_days)

# Dynamic fold creation
folds = []

step = max(3, n_days // 5)

for i in range(step, n_days - step, step):

    train_days = unique_days[:i]
    valid_days = unique_days[i:i+step]

    if len(train_days) > 0 and len(valid_days) > 0:

        folds.append((train_days, valid_days))

print("\\nGenerated Folds:")

for i, (tr, va) in enumerate(folds):

    print(f"Fold {i+1}")
    print("Train:", tr)
    print("Valid:", va)
    print()


Unique Days: [np.int64(48), np.int64(49)]
Total Unique Days: 2
\nGenerated Folds:


In [36]:

# =========================
# MULTI FOLD TRAINING
# =========================

test_preds = []

val_scores = []

for fold_num, (train_days, valid_days) in enumerate(folds):

    print(f'\n===== FOLD {fold_num+1} =====')

    train_mask = train_df['day'].isin(train_days)
    valid_mask = train_df['day'].isin(valid_days)

    X_train = train_df.loc[train_mask, features].copy()
    y_train = train_df.loc[train_mask, 'demand']

    X_valid = train_df.loc[valid_mask, features].copy()
    y_valid = train_df.loc[valid_mask, 'demand']

    X_test = test_df[features].copy()

    for col in categorical_features:

        X_train[col] = X_train[col].astype(str)
        X_valid[col] = X_valid[col].astype(str)
        X_test[col] = X_test[col].astype(str)

    cat_indices = [
        X_train.columns.get_loc(col)
        for col in categorical_features
    ]

    model = CatBoostRegressor(
        iterations=8000,
        learning_rate=0.02,
        depth=9,
        loss_function='RMSE',
        eval_metric='RMSE',
        random_seed=42,
        l2_leaf_reg=10,
        bagging_temperature=2,
        random_strength=2,
        verbose=500
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        cat_features=cat_indices,
        early_stopping_rounds=300,
        use_best_model=True
    )

    val_pred = model.predict(X_valid)

    score = r2_score(
        y_valid,
        val_pred
    )

    print("Fold R2:", score)

    val_scores.append(score)

    test_pred = model.predict(X_test)

    test_preds.append(test_pred)


In [37]:

# =========================
# FINAL ENSEMBLE
# =========================

final_test_pred = np.mean(
    test_preds,
    axis=0
)

print("Mean CV:", np.mean(val_scores))


Mean CV: nan


In [38]:

# =========================
# SUBMISSION
# =========================

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_test_pred
})

submission.to_csv(
    'submission10.csv',
    index=False
)

submission10.head()


NameError: name 'submission10' is not defined


# Optional Pseudo Labeling

After your first strong submission:

1. Predict the test set
2. Add confident predictions back into train
3. Retrain all folds
4. Average predictions again

This is often the key step for breaking leaderboard plateaus.
